# Role Prompting and System Instructions

Every chat completion has at least two layers of instruction: **what the user asks right now** and **how the assistant should behave in general**. **Role prompting** assigns a persona — tutor, analyst, editor, security reviewer — with tone, expertise, and boundaries. **System instructions** are the API mechanism that carries that persistent behavior across turns.

Well-designed system prompts reduce repetition (you do not re-explain format every message), improve consistency, and define scope (what the assistant will and will not do). Poor system prompts create conflicting rules, hidden defaults, or personas that fight the user task.

This notebook covers system vs user messages, provider-specific APIs, patterns for reliable system prompts, common personas, and live comparisons showing how the same user question changes under different roles.

Prerequisites: **01–04** in this section and chat API basics from **03. LLM API Integration**.

---

## 1. System Instructions and Personas

### 1.1 Message Roles in Chat APIs

Modern chat APIs structure conversation as a list of **messages**, each with a **role**:

| Role | Typical purpose |
|------|-----------------|
| **system** | Global behavior, persona, rules, output defaults |
| **user** | The human (or app) request for this turn |
| **assistant** | Prior model replies (conversation history) |

The **system** message is not a user question — it is configuration for how every subsequent user message should be interpreted. Think of it as the assistant's job description for this session.

### 1.2 What a Persona Specifies

A **role prompt** or **persona** usually defines:

| Dimension | Example |
|-----------|---------|
| **Identity** | "You are a senior data analyst at a healthcare company." |
| **Expertise** | "You specialize in SQL, cohort analysis, and experiment design." |
| **Tone** | "Be direct and professional; avoid jargon unless the user uses it first." |
| **Output shape** | "Use bullet points; max 150 words unless asked for detail." |
| **Boundaries** | "Do not provide medical diagnoses; suggest seeing a clinician." |
| **Process** | "Ask one clarifying question if the request is ambiguous." |

### 1.3 System vs Stuffing Everything in the User Message

**User-only prompting** (all rules in the user message) works for one-off tasks. **System prompting** is better when:

- The same behavior must persist across **many turns**
- Multiple endpoints or features share one **brand voice**
- You want to **separate** stable rules from changing user input (easier to version and test)
- The provider gives system messages **higher priority** in alignment training

Rule of thumb: put **durable behavior** in system; put **the specific task and data** in user.

---

## 2. Provider Differences

All major providers support system-level instructions, but the **API shape** differs. Your application layer should normalize these when building multi-provider apps (as in **03. LLM API Integration**).

### 2.1 OpenAI — `system` Role in `messages`

OpenAI Chat Completions accept a message with `"role": "system"` as the first entry (convention, not strict requirement):

```python
messages = [
    {"role": "system", "content": "You are a concise Python tutor."},
    {"role": "user", "content": "Explain list comprehensions in one paragraph."},
]
client.chat.completions.create(model="gpt-4o-mini", messages=messages)
```

Some newer APIs also expose a dedicated `instructions` field; the `system` message pattern remains the standard in this curriculum.

### 2.2 Anthropic Claude — `system` Parameter

Claude keeps system text **outside** the `messages` array:

```python
client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    system="You are a concise technical writer.",
    messages=[{"role": "user", "content": "Summarize edge computing."}],
)
```

Only `user` and `assistant` appear in `messages`. Do not send `role: system` inside Claude's message list.

### 2.3 Google Gemini — `system_instruction`

Gemini passes system behavior via config:

```python
from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="Summarize edge computing.",
    config={"system_instruction": "You are a concise technical writer."},
)
```

### 2.4 Comparison Table

| Provider | Where system lives | Message roles in history |
|----------|-------------------|--------------------------|
| **OpenAI** | `messages[0]` with `role: system` | system, user, assistant |
| **Claude** | Top-level `system=` string | user, assistant only |
| **Gemini** | `config["system_instruction"]` | user/model turns in `contents` |

Demos below use **OpenAI**; the persona principles transfer to all providers once you map the field names.

---

## 3. Designing Reliable System Prompts

### 3.1 Anatomy of a Production System Prompt

A maintainable system prompt often follows this structure:

1. **Role** — who the assistant is
2. **Goal** — what success looks like for the user
3. **Rules** — must-do and must-not-do lists
4. **Output format** — structure, length, language
5. **Scope / refusal** — topics to decline or escalate
6. **Optional tools** — when to call functions or retrieve docs (covered later in advanced topics)

Example skeleton:

```
You are AcmeShop Support Bot.

Goal: Help customers track orders and start returns.

Rules:
- Use only information the user provides or tool results return.
- If an order ID is missing, ask for it once.
- Never invent tracking numbers.

Format: Short paragraphs; friendly tone; under 120 words.

Refuse: Legal advice, medical claims, and competitor comparisons.
```

### 3.2 Avoiding Conflicting Instructions

Models try to satisfy **all** constraints. Contradictions cause unpredictable compromises:

| Conflict | Symptom |
|----------|---------|
| System: "Be extremely brief" + User: "Write a 2000-word essay" | Truncated or ignored length |
| System: "Never use bullet points" + User: "List 5 items" | Awkward prose |
| System: "Always cite sources" + no sources in context | Hallucinated citations |

**Fixes:**

- State **priority**: "User messages override format preferences but not safety rules."
- Keep system rules **minimal** — only what must hold every turn
- Move task-specific format to the **user** message
- Version and **test** system prompts like code (notebook 09)

### 3.3 Refusal and Scope Behavior

Explicit boundaries improve UX and safety:

- Say **what to do instead** of the forbidden action ("I can't diagnose; I can explain general symptoms and suggest seeing a doctor.")
- Define **in-scope** topics positively, not only prohibitions
- Align system refusal with product policy — vague "be helpful" fights strict compliance rules

---

## 4. Role Prompting Patterns

### 4.1 Common Personas

| Persona | System emphasis | Typical use |
|---------|-----------------|-------------|
| **Tutor** | Patience, checks understanding, analogies | Education, onboarding |
| **Analyst** | Data-first, caveats, structured insights | BI, research summaries |
| **Editor** | Grammar, clarity, preserve meaning | Writing assistance |
| **Coder** | Idiomatic code, edge cases, tests | Dev tools |
| **Reviewer** | Critique, severity ratings, actionable fixes | Code/doc review |
| **Support agent** | Empathy, policy adherence, escalation | Customer service |

### 4.2 Role in User Message vs System

You can say "Act as a lawyer" in the **user** message for a one-off task. Put the role in **system** when it defines the product:

- **System:** "You are Legal Draft Assistant; you summarize contracts, you do not provide legal advice."
- **User:** "Summarize the indemnity clause in the attached text."

Combining both is fine if they agree. Redundant conflicting roles (system: coder, user: poet) confuse the model.

### 4.3 Dynamic System Prompts

In applications, system text is often **templated**:

```python
system = f"""You are {product_name} assistant.
Today's date: {today}
User tier: {tier}
Respond in {locale}."""
```

Inject only what changes behavior. Huge system prompts cost tokens on every request.

---

## 5. Demonstration Setup

Replace the placeholder API key before running. The helper `chat()` sends an optional system prompt plus a user message.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)
MODEL = "gpt-4o-mini"

USER_QUESTION = "What is recursion in programming?"


def chat(user_content: str, system_content: str | None = None, max_tokens: int = 300) -> str:
    messages = []
    if system_content:
        messages.append({"role": "system", "content": system_content})
    messages.append({"role": "user", "content": user_content})
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.7,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content or ""


print("Helper ready.")

---

## 6. Demonstration: No System vs Persona System

Same user question — first with **no** system message, then with a **tutor** persona.

In [ ]:
TUTOR_SYSTEM = """You are a patient programming tutor for beginners.
- Use a simple analogy before technical terms.
- Keep answers under 120 words.
- End with one short check question for the student."""

print("=== NO SYSTEM MESSAGE ===")
print(chat(USER_QUESTION, system_content=None))
print("\n" + "=" * 60 + "\n")
print("=== TUTOR SYSTEM PROMPT ===")
print(chat(USER_QUESTION, system_content=TUTOR_SYSTEM))

The tutor persona should produce a gentler tone, an analogy (e.g. Russian dolls or mirrors), a length cap, and a closing question. The default answer is often denser and more encyclopedic.

---

## 7. Demonstration: Same Data, Different Roles

One paragraph of sales data — **analyst** vs **executive briefing** personas.

In [ ]:
DATA = """Q3 results: Revenue $4.2M (+8% YoY). North region +15%, South flat.
Enterprise tier grew 22%; SMB churn rose to 4.1%. Marketing spend $620K."""

ANALYST_SYSTEM = """You are a senior business analyst.
- Lead with metrics and comparisons.
- Call out risks and data gaps.
- Use bullet points; no hype."""

EXEC_SYSTEM = """You are a chief of staff writing for the CEO.
- Maximum 3 bullets, each one sentence.
- Focus on decisions needed, not methodology.
- Plain language; no jargon."""

user_prompt = f"Summarize this for leadership:\n\n{DATA}"

print("=== ANALYST PERSONA ===")
print(chat(user_prompt, system_content=ANALYST_SYSTEM))
print("\n" + "=" * 60 + "\n")
print("=== EXECUTIVE BRIEFING PERSONA ===")
print(chat(user_prompt, system_content=EXEC_SYSTEM))

Both summarize the same facts; the analyst version should emphasize YoY, churn risk, and regional split. The executive version should be shorter and action-oriented.

---

## 8. Demonstration: Editor and Coder Personas

Role prompts change **what gets optimized** — clarity vs correctness.

In [ ]:
DRAFT = "The report were finalized by the team and it show significant improvement."

EDITOR_SYSTEM = """You are a copy editor.
- Fix grammar and awkward phrasing only.
- Do not change meaning or add facts.
- Return only the revised sentence."""

CODER_SYSTEM = """You are a Python expert.
- Return a single function that solves the task.
- Include type hints and a one-line docstring.
- No explanation outside the code block."""

coder_task = "Write a function that returns True if a string is a palindrome (ignore case and spaces)."

print("=== EDITOR ===")
print(chat(f"Revise:\n{DRAFT}", system_content=EDITOR_SYSTEM, max_tokens=80))
print("\n=== CODER ===")
print(chat(coder_task, system_content=CODER_SYSTEM, max_tokens=200))

---

## 9. Demonstration: Scope and Refusal

A support bot system prompt with explicit **in-scope** topics and a polite refusal template.

In [ ]:
SUPPORT_SYSTEM = """You are ShopBot for Acme Store.

In scope: order status, returns within 30 days, shipping times.
Out of scope: medical advice, legal advice, competitor pricing.

If out of scope, reply: "I can only help with Acme orders and returns. For [topic], please contact the appropriate professional."
Never invent order numbers or tracking IDs."""

in_scope = "How long do returns take after I ship the package?"
out_of_scope = "Is this rash on my arm eczema? What cream should I buy?"

print("=== IN SCOPE ===")
print(chat(in_scope, system_content=SUPPORT_SYSTEM, max_tokens=150))
print("\n=== OUT OF SCOPE (should refuse) ===")
print(chat(out_of_scope, system_content=SUPPORT_SYSTEM, max_tokens=150))

Models may still occasionally over-help on out-of-scope requests. Treat system refusal as **risk reduction**, not a guarantee — add policy filters in production when needed.

---

## 10. Multi-Turn: System Persists Across History

The system message is sent once per API call but applies to the **full** message list. Demonstration: tutor system + follow-up user turn.

In [ ]:
messages = [
    {"role": "system", "content": TUTOR_SYSTEM},
    {"role": "user", "content": "What is a base case in recursion?"},
]

first = client.chat.completions.create(
    model=MODEL, messages=messages, temperature=0.7, max_tokens=250
)
assistant_reply = first.choices[0].message.content or ""
messages.append({"role": "assistant", "content": assistant_reply})
messages.append({"role": "user", "content": "Can you give a tiny Python example?"})

second = client.chat.completions.create(
    model=MODEL, messages=messages, temperature=0.7, max_tokens=250
)

print("Turn 1 (excerpt):", assistant_reply[:200], "...\n")
print("Turn 2:")
print(second.choices[0].message.content)

The tutor rules (simple language, short answers) should carry into turn 2 without repeating the system prompt in the user message. In production, **re-send the same system message** on every API call for clarity and provider consistency.

---

## 11. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Kitchen-sink system prompt | Conflicts, ignored rules | Split stable vs per-task instructions |
| Persona with no concrete rules | Generic "helpful assistant" | Specify tone, length, format |
| Wrong provider API shape | Silent drops (e.g. system in Claude messages) | Use provider-native field |
| Huge system on every call | Token cost | Template; inject only deltas |
| Safety only in user message | Inconsistent refusals | Put scope/refusal in system |
| Never updating system prompt | Drift from product | Version like code (notebook 10) |

---

## 12. Summary

**System instructions** define persistent assistant behavior — persona, tone, format, and boundaries — separate from each **user** task. **Role prompting** assigns identity and expertise (tutor, analyst, editor, coder) so the same question gets different useful answers.

Providers map system text differently: OpenAI uses a `system` message, Claude uses a `system` parameter, Gemini uses `system_instruction`. Design prompts with clear goals, non-conflicting rules, and explicit scope.

Demonstrations compared no-system vs tutor personas, analyst vs executive summaries, editor vs coder optimization, refusal behavior, and multi-turn persistence.

Next: **06. Structured Output and Format Control** — JSON schemas, parsing, and forcing machine-readable responses.